In [1]:
import re
import torch
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)


C:\Users\USER\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== GPT-style (decoder-only) =====
# ArabianGPT
# "riotu-lab/ArabianGPT-03B"
# "ArabianLLM/ArabianGPT-0.1B"
# "aubmindlab/aragpt2-base"

# Qwen (multilingual, very strong)
# "Qwen/Qwen1.5-0.5B"
# "Qwen/Qwen1.5-1.8B"
# "Qwen/Qwen2-1.5B-Instruct"
# "Qwen/Qwen2-0.5B"

# ===== BERT-style (encoder-only) =====
# "aubmindlab/araBERTv2-base"
# "aubmindlab/bert-base-arabertv02"
# "CAMeL-Lab/bert-base-arabic-camelbert-ca"

MODEL_NAME = "aubmindlab/aragpt2-base"


In [3]:
TASHKEEL_REGEX = re.compile(r'[\u0617-\u061A\u064B-\u0652]')

def remove_tashkeel(text):
    if not isinstance(text, str):
        return text
    return re.sub(TASHKEEL_REGEX, '', text)


In [4]:
df_pos = pd.read_csv("tashbeeh_dataset.csv")
df_neg = pd.read_csv("non_tashbeeh_random_sample.csv")

df_pos["label"] = 1
df_neg["label"] = 0

TEXT_COL = "البيت_الكامل"

df = pd.concat([df_pos, df_neg], ignore_index=True)
df = df[[TEXT_COL, "label"]].dropna()

# 🔥 Remove tashkeel
df[TEXT_COL] = df[TEXT_COL].apply(remove_tashkeel)

df.head()


,البيت_الكامل,label
0,أهم بشيء والليالي كأنها تطاردني عن كونه وأطارد,1
1,تثنى على قدر الطعان كأنما مفاصلها تحت الرماح م...,1
2,ولما رأيت الناس دون محله تيقنت أن الدهر للناس ...,1
3,مخضبة والقوم صرعى كأنها وإن لم يكونوا ساجدين م...,1
4,تنكسهم والسابقات جبالهم وتطعن فيهم والرماح الم...,1


In [5]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df[TEXT_COL].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)


In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,
    trust_remote_code=True   # 🔥 REQUIRED for Qwen
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    trust_remote_code=True
)

model_type = model.config.model_type
print("Model type:", model_type)

# ===== Decoder-only models (GPT / Qwen) =====
if model_type in ["gpt2", "qwen", "qwen2", "gpt_neo", "gpt_neox"]:
    print("Applying decoder-style PAD token fix")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.pad_token_id


c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\huggingface_hub\utils\_validators.py:186: UserWarning: The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
  warnings.warn(
Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at aubmindlab/aragpt2-base and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model type: gpt2
Applying decoder-style PAD token fix


In [7]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=512
    )

train_df = pd.DataFrame({"text": train_texts, "label": train_labels})
test_df  = pd.DataFrame({"text": test_texts,  "label": test_labels})

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test_df).map(tokenize, batched=True)

data_collator = DataCollatorWithPadding(tokenizer)


Map: 100%|██████████| 218/218 [00:00<00:00, 2069.32 examples/s]


In [8]:
training_args = TrainingArguments(
    output_dir="./arabic_qwen_tashbeeh",

    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,          # ⭐ keep only 1 checkpoint

    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,

    num_train_epochs=20,
    weight_decay=0.01,

    logging_strategy="steps",
    logging_steps=10,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,      # explicit & safe

    fp16=True,
    report_to="none"
)


c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [9]:
per_device_train_batch_size=1
gradient_accumulation_steps=16


In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}


In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\accelerate\accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [12]:
trainer.train()


  1%|          | 10/1080 [00:07<10:46,  1.66it/s]

{'loss': 0.7961, 'grad_norm': 28.244007110595703, 'learning_rate': 1.9851851851851855e-05, 'epoch': 0.18}


  2%|▏         | 20/1080 [00:13<10:30,  1.68it/s]

{'loss': 0.7721, 'grad_norm': 22.581789016723633, 'learning_rate': 1.9666666666666666e-05, 'epoch': 0.37}


  3%|▎         | 30/1080 [00:19<10:21,  1.69it/s]

...

                                                   
100%|██████████| 1080/1080 [12:14<00:00,  1.72it/s]

{'eval_loss': 0.8038138747215271, 'eval_accuracy': 0.6788990825688074, 'eval_runtime': 1.2781, 'eval_samples_per_second': 170.572, 'eval_steps_per_second': 43.034, 'epoch': 19.82}


100%|██████████| 1080/1080 [12:18<00:00,  1.46it/s]

{'train_runtime': 738.3798, 'train_samples_per_second': 23.619, 'train_steps_per_second': 1.463, 'train_loss': 0.5205026690606718, 'epoch': 19.82}


TrainOutput(global_step=1080, training_loss=0.5205026690606718, metrics={'train_runtime': 738.3798, 'train_samples_per_second': 23.619, 'train_steps_per_second': 1.463, 'total_flos': 158919458881536.0, 'train_loss': 0.5205026690606718, 'epoch': 19.81651376146789})

In [13]:
preds = trainer.predict(test_ds)
y_pred = np.argmax(preds.predictions, axis=1)

print("Accuracy:", accuracy_score(test_labels, y_pred))
print(confusion_matrix(test_labels, y_pred))
print(classification_report(
    test_labels,
    y_pred,
    target_names=["Non-Tashbeeh", "Tashbeeh"]
))


100%|██████████| 55/55 [00:01<00:00, 40.73it/s]

Accuracy: 0.6972477064220184
[[77 32]
 [34 75]]
              precision    recall  f1-score   support

Non-Tashbeeh       0.69      0.71      0.70       109
    Tashbeeh       0.70      0.69      0.69       109

    accuracy                           0.70       218
   macro avg       0.70      0.70      0.70       218
weighted avg       0.70      0.70      0.70       218



In [14]:
safe_name = MODEL_NAME.replace("/", "_")

trainer.save_model(f"{safe_name}_tashbeeh")
tokenizer.save_pretrained(f"{safe_name}_tashbeeh")


('aubmindlab_aragpt2-base_tashbeeh\\tokenizer_config.json',
 'aubmindlab_aragpt2-base_tashbeeh\\special_tokens_map.json',
 'aubmindlab_aragpt2-base_tashbeeh\\vocab.json',
 'aubmindlab_aragpt2-base_tashbeeh\\merges.txt',
 'aubmindlab_aragpt2-base_tashbeeh\\added_tokens.json')